# 4. Variable Stars - Under construction!

In this notebook tutorial we show how you can include variable sources in PlatoSim and extract their light curves.

### Setup notebook

In [ ]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# To interact with the plot use
%matplotlib notebook

### Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PlatoSim
import platosim.plot            as pt
import platosim.utilities       as ut
import platosim.referenceFrames as rf
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_notebook
setup_notebook()

In [ ]:
# We use the default input YAML file and configure is later
outputDir = os.getcwd()

---
## 4.1 -  Include a variable source for a target star
---

We here show the simplest example on how to include stellar variability for a single target star. First let's create an instance of our simulation class:

In [ ]:
# Set up a Simulation object
sim = Simulation("output_example1", outputDir=outputDir)

To visually make the effect of variability clear we here generate a synthetic sinusoidal signal with an amplitude of several orders of magnitude on (very unrealistic) short time scale: 

In [ ]:
# Simple model of variable source
A, P = 2, 10000
time = np.arange(0, P, 25) 
dmag = A * np.cos(2*np.pi * (time/P))

In [ ]:
# Show plot of variable source
plt.figure(figsize=(8,4))
plt.plot(time/3600, dmag, 'k-')
plt.xlabel('Time [hours]')
plt.ylabel(r'Relative magnitude, $\delta m$')
plt.tight_layout();

We can now use `sim.createVariableSourceFile()` to create an ascii file that PlatoSim can read:

In [ ]:
# Automatic catalogue file creation
variableSourceFile = outputDir + "/varsource_example1.txt"
sim.createVariableSourceFile(time, dmag, variableSourceFile)

Assume now we just want to simulate a image time series of a single target star and inject the newly created variability for this star. Then we can use `sim.createVariableSourceList()` to create the ascii file that tell PlatoSim which star a given variable source file should be associated with. Hence:

In [ ]:
# Automatic catalogue file creation
starID = [1]
variableSourceList = outputDir + "/varlist_example1.txt"
sim.createVariableSourceList(starID,  [variableSourceFile], variableSourceList)

Again, just like for the creation of the star catalogue file, while using `sim.createVariableSourceList()` the the inclusion of variable sources is activated and the variable source file is automatically set in the YAML tree. If you need to change these simply use:

In [ ]:
sim["Sky/IncludeVariableSources"] = True
sim["Sky/VariableSourceList"]     = variableSourceList

We now use the  methodlogy from the previous star catalogue notebook to place a star in the CCD focal plane and create a star catalogue file:

In [ ]:
# Select subfield size and location
sim["SubField/NumColumns"]      = 10
sim["SubField/NumRows"]         = 10
sim["SubField/ZeroPointRow"]    = 3000
sim["SubField/ZeroPointColumn"] = 3000

# Define catalogue
row = np.array([5.0]) + sim["SubField/ZeroPointRow"]
col = np.array([5.0]) + sim["SubField/ZeroPointColumn"]
mag = np.array([11.0])

# Automatic catalogue file creation
starcatFile = outputDir + "/starcat_example1.txt"
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, starID, starcatFile)

We can now run simulation of the number of exposures that covers one variable period:

In [ ]:
sim["ObservingParameters/NumExposures"] = int(P/25.)

In [ ]:
simfile = sim.run(removeOutputFile=True)

In [ ]:
fig, ax = simfile.showImage(clipPercentile=1, imgScale="clip", fontSize=20, 
                            colorMap="cubehelix", colorBar=True, showGrid=True);

Draw the slider from side to side to see the impact of variability.

---
## 4.2 - Variability of both target and contamint stars
---

---
## 4.3 - Generate noise-less light curves
---

### Exoplanet transit

### Stellar granulation

In [ ]:
time =  array(outputFile["/StarPositions/Time"])

In [ ]:
timeScale = array([2060.0])       # sec
varScale  = array([520.0])        # ppm

More decent values for Sun-like stars can be obtained from Pallé, P.L., Roca-Cortés, T., & Jimenez, A. 1999, ASP
Conf. Ser. 173, Stellar Structure: Theory and Test of Convective Energy Transport, ed. A. Gimenez, E.F. Guinan, &
B. Montesinos, 297.

In [ ]:
granulation = noise.redNoise(time, timeScale, varScale) 

In [ ]:
fig = plt.figure(figsize = (10, 10))
plt.plot(time, granulation)
plt.xlabel("Time [s]")
plt.ylabel("Signal [ppm]")

### Combined signal

In [ ]:
flux *= signal
relativeFlux = (flux - flux.mean())/flux.mean()*1.e6     # ppm
relativeFlux += granulation

Bin the time series to make the structure inside more clear

In [ ]:
binnedMean, binEdges, binNumber = binned_statistic(time, relativeFlux, statistic='mean', bins=100)
binCenters = (binEdges[1:] + binEdges[:-1])/2.0

Now plot the original time series as well as the binned time series.

In [ ]:
fig = plt.figure(figsize = (10, 10))
plt.plot(time, relativeFlux, c='b')
plt.plot(binCenters, binnedMean, c='w', linewidth=3)
plt.xlabel("Time [s]")
plt.ylabel("Relative Flux [ppm]")